# A classifier on a truncated frozen backbone

Part 8.4 of the 2026 paper built a 5-class classifier on ResNet-50 cut off after `layer3[4]`, with the whole backbone frozen. Here is the same task on a smaller pretrained network.

```python
class TruncatedClassifier(nn.Module):
    def __init__(self, pretrained, keep, num_classes):
        super().__init__()
        # Your code

    def forward(self, x):
        # Your code
        return logits
```

- `pretrained` is an `nn.Sequential` of image layers (convolutions, batch norms, activations, pooling, possibly nested `nn.Sequential` blocks). It maps a batch of shape `(B, 3, H, W)` to feature maps `(B, C, h, w)`.
- `self.backbone` must consist of exactly the first `keep` layers of `pretrained`, **the same layer objects** (so it uses the pretrained weights), and all of its parameters must be frozen. The layers after position `keep - 1` are not used.
- After the backbone, average each channel over all positions (global average pooling), flatten to `(B, c)`, and apply a new trainable `nn.Linear(c, num_classes)` with bias, stored as `self.head`. Here `c` is the number of channels the truncated backbone outputs; your code must work it out.
- `forward(x)` takes `x` of shape `(B, 3, H, W)` for any $H, W \ge 16$ and returns logits of shape `(B, num_classes)`.
- The only trainable parameters are the head's.

**Example.**

```python
pretrained = nn.Sequential(
    nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(),
    nn.Conv2d(8, 16, 3, stride=2, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
    nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
)
model = TruncatedClassifier(pretrained, keep=5, num_classes=5)
model.head                                    # Linear(in_features=16, out_features=5, bias=True)
model(torch.randn(2, 3, 32, 32)).shape        # torch.Size([2, 5])
sum(p.numel() for p in model.parameters() if p.requires_grad)   # 85
```

**Restrictions.** Only `import torch` and `import torch.nn as nn`.

**Constraints.** $1 \le$ `keep` $\le$ `len(pretrained)`; the kept layers always end with at least one convolution somewhere. Time limit 1 s per construction and per forward call, CPU.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem6_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem6_submission.py
import torch
import torch.nn as nn


class TruncatedClassifier(nn.Module):
    def __init__(self, pretrained, keep, num_classes):
        super().__init__()
        # Your code

    def forward(self, x):
        # Your code
        return logits


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem6_submission as submission
importlib.reload(submission)
# e.g. submission.TruncatedClassifier(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9W41z2zaW/1dw7syazMqMrKS5nlP31nHdNjv5mjg73Rvbw0IiJLGmQIagYrte/e/3ew8ACUqyE6d7152NRRB43+/hvQfwdifPdg7EznJ/uFcNn+0MxM5CNRJDt5tvmrwpFI0diUkhjcmnuapFqYUUTb3UE9moTEzr8g+lxVhOLselVrSuUMaU2gMrhv/FaHK9bJTB6GiIxyyfTvPJsmhuMPIEA0o3Nf3e+eBBH7c4eX2ZLQuVarlgkqq6HBdq8Sw1y/EixzwgxCxZFOWVytJ8UZV1Q9jOdpqynsx3LvB2LLVWxOWZe0qLsqxo1lQWRhHLyjRpky9UuWzSlthGzgJQybRW6o9cz1hGPOLF0Y1UZVn0plj6mQxTLusJcyEzWYHTvdFw9Gzv/f7eu+9oQSXrZoTXgKpWeJ7mBQvudmcuaw3hJhVL6pz/94sqKlUb0cxlI5xYxKyWGQ1aOYjoN6hpIdxyP/rotzg51+f6yE0XuRGTUjdQxeOlzhvz+Hv688NjBxXPVZ798NjOBhGJeNmITE1zQBUkOjGFHBrowghSVMaD6SNLWyMvFX6pc93pTFixiN8w9FsiTv559Prdq5NTUeSmIZaUBUvAjDDz8kqT/b1fahF9yk0OquLn4pTANQI6MEJ9UvUNLwJrP8m8WNZYWcvcKEGP4ipv5jBgwDNypsRVnTdgWEzLWrxQ+kAYeSOuiNwrafC21LOB0ARVZMtFJeZ5lmF6rqtlY0h6VgvaiXRSVjftA9kRydc9aqy/EYCqKxr9RhyXhRyL43f/MELWShgy3ZpkpYmUXSNej8S7uhwIU4qinMiCIMKojFgsIZSxIqFCRCW5Uo7XRU5yGIP/OVQ5lROYXkKYRsmQlCvFbEn6jyxmA1iF2mvmtZKZMJWCwjCrLpezeXEDaymmArSAiklZk6BrRUJvRD4lAvHITub4cF5DQjl9d3LyY/rT0fGHt+/FIWEnhs81RxFWQ3RyPVEVWUp8cK4F/qvwys6CQUHiKotgi9lALMzMzwFeXZKQdeZG6L9OuRHN7YCkuoqu/VpEl2CN1xC5ZjA6Bf+5No3UExVdD+z75IPSpqzjYDmjVc2y1uI6yRBAJ/MoTibVEv+ymqPYTlbMpnjJ6E7quqwDKJblAJauEmlkXcsb0N3xYeayUtGsbAYwSk3/wj4Pz3dg2Qig5zueMgNZN8uqUBFxjvlxwksdLSxTzPGTCFY8ENPznVsCuILCjcUlbg3ijrqu1IRC/G0wfwV0HWWTojT3UTYQdVMWh/tq79lASPfzO08vPOsKJLfEStPcgNFpUUoijMYZZ/9FwMzM8kcsXdmfd7BjGXDT4228+VfMXosACskNwlveYDXoKIooFogUZIXhuyv3LkRPkVTmHAk1rcn1tAXuDJn0XRROiBCGExf948RF/4SGp+oaEmNBEHELeU1/5NhEM7EnruI4JthgNP9DCYU9TQzJ+ba4Sl9OHOjEJ1kgPoiokPWMwq4igxW3+HOQPJmt4p7u1celLL7AKvtOZacS0TpjSx/ELFEQIZumdu/Pd1gdHZAt5hIYSDfn/9wsQtNg+lMnBxC34UyhVFsIpJWDdYphDEQtM9/BQDC+xav/qHuk0SwM9ZSRm5RdhNVBv8yd+mCEgUa6JRv00yhE4QDHScrpV5r2yJkh/JHmHNZdP2nXQ+1TmhotK+zkzf9vaC6Qmz4wKAOlt8prMkne23dibIkZe2+fnijLJxAkJS5xSE+fuR51BDDKlKoOkfOqwMAcgR8gvnXyNgHEvS2EhhKCye+ue4KHWiI5EOOvl7v8IrnbKdYvGN/Xil2S2BmO6YcCh0haJCYax9tiDZZ3cWbr8jZOjN1P1m3fs4mDgY12sGwd6srDYQhB8lKmiyWcAtlNNNUD8QjhFJ7x6NHlFf3ylNjU8Rg7gJhqxjylBJV4QI6DLHAG95IaKWM9Wy6QlGMLEVUhJyrxWSfv+uRPiItnnWfJmPNZSQsI44WdieyMNg8drRNkX9OSHIYMhk1MSxVSGVWjrIn+yKvIrmEcPQPneNIalwsiNyhxYIuZogSfarjMpswdL9/c5uKvYn/1XFyV9aWtKslmBakP+WgbL52QQXwnYUricx0ZRakgqPpSKT8PRYzoGqbcWzJpriaxHxrk3lTqQkCc6+62K5G82xqKUL0FiJpKMBRIC0MscR7+Wk7EBAhoCZi0WTdVK5zYU3KA6mJvvMwQSWmYc3Ck5TkVUMsK7iSaq3yiCCAklGOHLqfOg0Gzl6hhE6KRKfIl2r4JAlYs8S4RH+hFXmMYzBS7hIL2BYuMJKyynlk1Q8omASFBdTlNGYyqfbT5jCllzfbFyFEal444fg+FU6J4LMKyoTPJlEyxJm+IRqHdQYlA8/2hg7QWisaoaS67ITkaiMsRsPXjI5M8WBvscfJ5UTCdJIgRi2EUDLMYULFFGW2xd4gjzDdDhtiPmrJku0HxnzUHyWi6QpYPK1PPXdXVGjEbE9yOAYmlpnr+1onGreMaEoWdV/GW/xglKkuUwVOY1owSWFtk3jo9MSDyjnsdtNYzeKfKvMK6AgeazMoFpD2VML+0nXjO/ZulkTIvU+4w9LocP9MIPACUdc2DnIPGPC8y8roJdUZQMOfkCzTz76dv34BWzZ5jmgwUsq9WN80cI3svhccjvnctjjTL6x/E9x0OvMQzvPJfPNb8sFHyw9mC8h+KvW6KfNwO2T8YSZZNXnTDZfvzd1N2TYJKNvNwuclnWnbLzI1Zay/4hxr7AjXh2hFYsaZmge01mEkONrVqwx0GLvNmr1CYJvaTZ8m+y+nMAfWJFtTEG3BIQXXhNh8f/CqZQ8qzc+1xJNO8gEX7x+h8B2SXtaIK0HVakIkmj/6hLzX1cExZUEel5A4AEOYVaS155DLFX07en8BznCySd/gbpSm1wdI0ThC4aDmyukrWoOxcQyoJTU5grapuoiGcAekDgYkBcFvvi3tB4hvs1R/lgTh5OhxxJ+zVr0f/c5q+OHrz5uRHUHB7vpO6XmKaEjNgzOvTPqprNXG/kOfbX5NyUYHW853VuX719u279M3bH09OAS6CrSQ/lfBV+vErDFfZn0fmRk/aF6+QQx4Dhn06VcHDj0gyu6eflab9uaxPrqs4bLF8sL2Yz3VZUolKbxGRjS0XiDfYtlTrsVwqekBhN4Iym0k6mavJpYlMPSEVN7K/3763/k7ZMDYrkmN5aYWzMDMI5jnCImWAtsfHbmOa3VY7nAy5ViJtVE2d27Zib4uym9DZxbZstqkVEjuWEqwE9TUI7eehpzeoz685D6WmnNrMD88c2QewFm4PO+oPKEwaXu5K5FLbOHOrEvqry9UB/cZsFD8rR6DrTdOWAHklCMzwk7WGNcno7MIROpbtOO+Tjd9yuB2aLg0DC8bH8IfU9krdOJk4Ukb4bp7BVbibbhMFCxgQ7PyIKhpKuvMxtenBziXnbqpIyLAdAm6XBzJfyGaxLMKRKXdHEKopNkP4V7K4jEgZa1t3kKbTdGvPtjRYryraVJYmJkzuweYG5loqyJcjyZMoKV9Qa8E+JsAGQVLmGC2QeSIcUTFHwBdMrFVEfLB9bwx0kcgsczjizcmdboJ5iamKnNRNOM/29i+2LOyUlyyrjLLuLWvDIrG4X4w/IeqtcwOFQlcsR9d4B/udQ22RJa9wgsTvPyHFdQkC2poUAtH1JLDNAtaW3iW8r1rKfD5U6Efed9Z5XzMIpoMc7Yugvgki8n0A8+xurrZP2oax26zWcbLjJ7KqlM4i3/Gxi853bMiz4f2/+/LahoV3wRe5fut2saPl7MjQHhTb3sragqR0817L5jVSxg2btjHogcSBsMAgA5CUJDpQd4Z/nwVYL7EmT6PsEDQv+b2k8pQDbBTgiePt0eWvtKNAMB7U7e5A7Fog3qPIAXeBD3XvbJdasavt/cRt9FPLYgv5xtPeAfumPUEcS21ItpO52yE4Few2CG4/K4ki9p/8N35uq9HiSt6YPXvSikoGeWHeAiKjdNBKXdwMPE5ULt1WE1krsE0kl0pR0nEjCjVDaX5FaZPgmUmSxMFmhqRMZCUpxQKe51xfWi1Ewab5F94cg13Ykus2X5jhv0QU7KV/Eb28MDAhYHiw6YACI5x8CH5k4i22A8jxn9GwLj2O9ki20zIo7zOfWidKy8rmk/1uG023Trbmel/CMFWUsIu/CQ+bStbAb0RkMyeH4Wx4ERr3FvYfIoIO633Mc3DbYJlHv45hmyaxsfcYxTC5MRl1reZKcxHb5o686t/Lv+NsdU+lPq1zgCluUs5kI4BAoZtSNdXP6N9qtcd0ZspM6pyrClHBVOHfM+FOonlC7o+vqWCDuyqaZ/iwvt9hGlNbxdetCepm/G7SZhypJE3bFyj5XLJJwJFpTnlTnxIegICmplR+Kp/zUeXXsuCy0itqmVDDiqIs9yad0eFfSseCrL03RKpgg6HZfJzWke+ESScmfEKigvMRyv5Xt4x1RfO9sBfIxKO2FkOkXa9yqZKV9ezTWZsielbunDm6CEpiJylKwA9FO+WJT9dh9xinhkNSlDIzUcREPCazIZ/gVsQOwcOeRZ2MyAc7lE8E0RETTvDS4G7iodj0C1swnrtrN+d8x8Z0j7YxY03WVdJkiTTwpoQKQvhnHbQLIueOUrR1dMpgyQXNmaXqgncLemIvuQjTWV/S+XzWcN9xA20YJbhrEbE46XKIiez0cJe3VuJbnlMQnZfJaUPtk5dvvfBskyexfyL3dPry56NX718PXJXupiKcpdzc9afAQSzrX14iWY6GnpZ+dcyXYLpmFdSZ5bWC81llRKB0Pc3yNKomJxS1J/Plh5evT96n70+OXg1a6tbSHFMpsp5+Kyyh0ZR6M7azQ91JPjIJOAoufVnT6BpzUFMQqtbwLceb6BwsRkioI/pnfSE8xs4zZ/SaC4YLa/ibLLEPqTqhHlBql1Hkib9CcMN+b8K1XXotCWeIzjkubDSzPFIE5qjGEbcpy0s+UxC3Xh8HyZA6wD50OSwvpFFth8j2QLqW2ItXJ8PhfkfAg9i4i2J3MMRk0kkM3RArnaJcEnTPftS69QbobYLycYXwwp7JovgkPkIB/nQ4HB5cfLUj813Ffh+Hh8L7HXaOPyv2J5rgZWBfxZ9XbierrFQ2abKX7bC7EIhVWLc/kIXZw3zyfMde+iMvbLeM9h5gy7e7R/h535sFzje7y5csNDfLXfdzjS1f5dkpFBn81UGfxLuwR3sNJQ0RCpWpDez0i2L7J1kbjyPJG7UwEd+e0f1dgaOqP/enAxQJMqJpHDa8ECXoEJJTEsK4ljzzdkzWXy8Rt6xRULbCtwi0Z20t4FJ4zvVShfqb2A3WxsMDBnL27cEF4jefC4Na16UWHB7PdxxoPznARm+pG0xlz7YyxcVbe+hDQ8NkuHrImdjmxYev2HX+7M7jDuc2o/JXhTSnhLNOcuSplHhv3t3YiODdYpt7WB/nYF2UeubPZdZjtohybW+ZcTY/oAJiKmvhTwbjMAg45Hy4sdbT/pMMtzRTer3lrsqD9pJ/Fy3hPQM+r8i+cA/pYLUGTvBqWHEW3XVQOxBPtm1vLp298AVZ3Qa2B29F7QKbqx5S6Rh1uet9WSkHlXFZFtEGYfYdgWp6lkvgmh64doUvUO7cUvha81T4gsdGtzSl+oYOyg788QCVO/Zkt90s6Ey3f9cofEo0W4/Wdx/bDfiIe2Avnw7cXRAiqL09jnjv4ra6louK4t9FcHHRjqWaDkj8VSh7R19quuVD59HRsF8za52cqo9LOkNBih7cftHJcak/jbLoyUB8N6BPGSqZZcjIDvcRgfH6vXr1jygebFuC+fvPeA2dcGXqcLSx+gU1zt6U9QLT9599FiKDG90Psrc8ONgbY7e/vPmMWPb9DQ+/IMoRkujez+b5WV9mHZF2RUeiCUgciHEuzSHvR5sSKEMO7lNQTy2QybcdipGF8Vpev4O7YAYNWFYwcR/y2m8H6Gn0FP9vR+jp6TCcQ0/PeE4gyytUbOmyiiDM7t5lkzS184h2O3RGb688RKEQ1+6+PNk4BoCe7Gq6UqEja3xM7dM49rO+EbP8EzccuZMLMeIfvQfBf6J7TkhI6MTeVtIGsSQ4Heay2masC3kJT7tUCvvP8UAgBVJwQ7pkbA6jiA0OIooDVik7xqKW10AeXY+iwCx3lWs/GXKUTrZ8/kMO37To4+3XWBme1Sxnj+4yK5O64u7YOlj7FQVw2jPzdmX/Cnh4U9YlnQ4VNQ/tR0+UK1GvIr4bv4GqlP2gpf10Clzx3Te6kKyKadKBcwRc0s56yKfoJLmzAxLBBR1oI+piYzKtE/AFYjvRdtI9rG2Tma9Cab7US6GbfhOubqegxklFndqPbIcfyQ7tDHqs6JHvYAcmGXLbp8F/rkKRd9IUN/aSGt9FuyWOVjCoGzqXLqfYcxQ7CfVLaRrdK7SvRTn+na6nxK14ltodgZ9VHVkkJwJ60JfTRXir358Tb2HSwgzYtKz0pNjXLFFpjZk7+44VOW3oLmBpck6IiCIkEft92yJJEzEVUuGPS+TChsPAOvLtutykopW367C7T/MiU0KG3VrsqZ+QL4UY3aVWT9ycvgc6FBv2Pud7mc7Wt/sITWEveAUdytpaFA0mFNc9ZQRgzT/IARivNxapOzA2VhKEQICNOy76omhKlVPnQi5w/qHq0kT7HDhdEHOfzJzthwbD9COzmSrZ8Pdk8JlJxxm128JXx33WmCswblmJbtehrQbidgPOKg7u2PuVE5p6vOoUpbk1tYgq+upI0dHZPXZLpeiapfW+dbFcPRLHyKDXWLjVK8FuSbVvYErhdwDt0pX1XGabjvk6csNDvq56/mUgfiWKeTsJVHZNFWawwT1hPdHsAUzT3Xw6tDPam1BR3EtXfgFRv4Y9kM+YSXetKNyXnANcr1UjpCvTt6vrmA9DoyxfHEZkUesnvvTxBlV+vPRv1n6uVD6bN8kHCK/1lLX71YGHgby1W/j2vJQ/WKGDIfvJCmmTRHbcCxQDdw379pfV9e2v0CwW0HP/gxjn94xpl1/tDgRCTv8TGQu9/2mM/ZaJF3Yfs9yBuihnub3x5D8Se9p9JPZtkEyFmTyX8k5hQYIS5PSc7H3LBw1z5V+IiPRz+G3/EyYGPIHXcbROsTnln7jflZZ1Sl/wfgk2m20jONovQ7l0FxO66s5bgBSUrTr8o5jYXc+bRERfGY++i72xBKi6pJwxPb0X00JeCyL7M9j2/5Ny2DhelwQKLSg2ZZTmDtZDeryce/RYIHaeCcS+jZCnXAB9jm3KyocbeEi5V/MSqvVplMX1rIeLEvNnTyk1RRW6Wv0vmFROug==")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
